In [ ]:
# Aleander M. Aquino
# Doctor of Engineering Cpe
# BIOINFORMATICS

In [4]:
import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt

# ==========================================
# 1. LOCAL DATA PREPARATION (PetImages)
# ==========================================

# dataset path
desktop_path = r"D:\DOES-193-DIT-DENG11G1---\DOES 1C3-DITDENG11G1 - Bioinformatics\PetImages"

# Dataset Parameters
IMG_SIZE = 150
BATCH_SIZE = 32

# Clean corrupt / unreadable images (common issue in Kaggle's PetImages dataset)
num_skipped = 0
for folder_name in ("Cat", "Dog"):
    folder_path = os.path.join(desktop_path, folder_name)
    if os.path.exists(folder_path):
        for fname in os.listdir(folder_path):
            fpath = os.path.join(folder_path, fname)
            try:
                fobj = open(fpath, "rb")
                is_jfif = tf.compat.as_bytes("JFIF") in fobj.peek(10)
            finally:
                fobj.close()

            if not is_jfif:
                num_skipped += 1
                os.remove(fpath)

print(f"Removed {num_skipped} corrupted images.")

# Load training dataset (80%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    desktop_path,
    validation_split=0.2,
    subset="training",
    seed=1337,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)

# Load validation dataset (20%)
val_ds = tf.keras.utils.image_dataset_from_directory(
    desktop_path,
    validation_split=0.2,
    subset="validation",
    seed=1337,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)

# Rescale pixel values from [0, 255] to [0, 1] & optimize performance
normalization_layer = layers.Rescaling(1.0 / 255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y)).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y)).prefetch(buffer_size=tf.data.AUTOTUNE)

   

Removed 0 corrupted images.
Found 23422 files belonging to 2 classes.
Using 18738 files for training.
Found 23422 files belonging to 2 classes.
Using 4684 files for validation.


In [5]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = 150

# ==========================================
# 2. BASELINE MODEL (Manual Tuning)
# ==========================================
def build_manual_model():
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.MaxPooling2D(2, 2),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

print("--- Training Manual Baseline Model ---")
manual_model = build_manual_model()
manual_history = manual_model.fit(train_ds, validation_data=val_ds, epochs=10)

--- Training Manual Baseline Model ---
Epoch 1/10
586/586 [==============================] - 234s 381ms/step - loss: 0.6401 - accuracy: 0.6483 - val_loss: 0.5520 - val_accuracy: 0.7186
Epoch 2/10
586/586 [==============================] - 369s 629ms/step - loss: 0.5179 - accuracy: 0.7475 - val_loss: 0.5282 - val_accuracy: 0.7385
Epoch 3/10
586/586 [==============================] - 368s 627ms/step - loss: 0.4197 - accuracy: 0.8109 - val_loss: 0.4859 - val_accuracy: 0.7792
Epoch 4/10
586/586 [==============================] - 341s 581ms/step - loss: 0.3189 - accuracy: 0.8610 - val_loss: 0.5144 - val_accuracy: 0.7837
Epoch 5/10
586/586 [==============================] - 117s 199ms/step - loss: 0.2224 - accuracy: 0.9087 - val_loss: 0.6623 - val_accuracy: 0.7724
Epoch 6/10
586/586 [==============================] - 101s 171ms/step - loss: 0.1613 - accuracy: 0.9384 - val_loss: 0.6222 - val_accuracy: 0.7763
Epoch 7/10
586/586 [==============================] - 98s 166ms/step - loss: 0.1278 -

In [6]:
# ==========================================
# 3. HYPERPARAMETER TUNING (KerasTuner)
# ==========================================
def build_tunable_model(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)))
    
    # Tune number of Conv layers (2 to 4) and filters
    for i in range(hp.Int('num_conv_layers', 2, 4)):
        model.add(layers.Conv2D(
            filters=hp.Choice(f'filters_{i}', values=[32, 64, 128]),
            kernel_size=3,
            activation='relu'
        ))
        model.add(layers.MaxPooling2D(2, 2))
    
    model.add(layers.Flatten())
    
    # Tune Dense units & Dropout
    model.add(layers.Dense(
        units=hp.Int('dense_units', min_value=64, max_value=256, step=64),
        activation='relu'
    ))
    model.add(layers.Dropout(rate=hp.Float('dropout', min_value=0.2, max_value=0.5, step=0.1)))
    
    model.add(layers.Dense(1, activation='sigmoid'))
    
    # Tune Learning Rate
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Initialize Hyperband tuner
tuner = kt.Hyperband(
    build_tunable_model,
    objective='val_accuracy',
    max_epochs=10,
    factor=3,
    directory='kt_dir',
    project_name='cats_vs_dogs_tuning'
)

stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)

print("--- Searching for Best Hyperparameters ---")
tuner.search(train_ds, validation_data=val_ds, epochs=10, callbacks=[stop_early])

# Fetch and print the best hyperparameter values
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"""
Search Finished! Best Hyperparameters:
- Conv Layers: {best_hps.get('num_conv_layers')}
- Dense Units: {best_hps.get('dense_units')}
- Dropout Rate: {best_hps.get('dropout')}
- Learning Rate: {best_hps.get('learning_rate')}
""")

Trial 28 Complete [00h 07m 01s]
val_accuracy: 0.8257899284362793

Best val_accuracy So Far: 0.8377454876899719
Total elapsed time: 03h 46m 28s

Search Finished! Best Hyperparameters:
- Conv Layers: 3
- Dense Units: 256
- Dropout Rate: 0.30000000000000004
- Learning Rate: 0.0001

